# Cinemática Inversa Numérica

In [45]:
import numpy as np
from dataclasses import dataclass
from typing import Self

## Versión inicial inexacta

In [46]:
@dataclass
class Pose:
  rotacion: np.ndarray
  translacion: np.ndarray

  @classmethod
  def identidad(cls) -> Self:
    return Pose(np.eye(3), np.zeros(3))

  def __mul__(self, pose: Self) -> Self:
    return Pose(
      self.rotacion @ pose.rotacion,
      self.rotacion @ pose.translacion + self.translacion,
    )

  def distancia_cuadrado(self, pose: Self) -> np.float64:
    diff = self.translacion - pose.translacion
    return diff[0]**2 + diff[1]**2 + diff[2]**2

def DH(theta: np.float64, d: np.float64, a: np.float64, alpha: np.float64) -> Pose:
    cth = np.cos(theta)
    sth = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)

    return Pose(np.array([
        [cth, -sth*ca,  sth*sa],
        [sth,  cth*ca, -cth*sa],
        [  0,     sa,       ca],
    ]), np.array([ a*cth, a*sth, d ]))

In [47]:
@dataclass
class Dimensiones:
  d1: np.float64
  a1: np.float64
  a2: np.float64
  d4: np.float64
  d6: np.float64

class IRB140:
  def __init__(self, dim: Dimensiones, ABase0: Pose = None, A6Tool: Pose = None):
    self.A01 = lambda q1: DH(q1, dim.d1, dim.a1, -np.pi/2)
    self.A12 = lambda q2: DH(q2,      0, dim.a2,        0)
    self.A23 = lambda q3: DH(q3,      0,  0,  np.pi/2)
    self.A34 = lambda q4: DH(q4, dim.d4,  0, -np.pi/2)
    self.A45 = lambda q5: DH(q5,      0,  0,  np.pi/2)
    self.A56 = lambda q6: DH(q6, dim.d6,  0,        0)

    self.ABase0 = Pose.identidad() if ABase0 is None else ABase0
    self.A6Tool = Pose.identidad() if A6Tool is None else A6Tool

  def cinematica_directa(self, q: np.ndarray) -> Pose:
    q1, q2, q3, q4, q5, q6 = tuple(q.tolist())
    A06 = self.A01(q1) * self.A12(q2) * self.A23(q3) * self.A34(q4) * self.A45(q5) * self.A56(q6)
    return self.ABase0 * A06 * self.A6Tool


In [48]:
def funcion_costo(pose_origen: Pose, pose_destino: Pose) -> np.float64:
  return pose_origen.distancia_cuadrado(pose_destino)

In [49]:
irb140 = IRB140(Dimensiones(352, 70, 360, 380, 65))

In [50]:
def cinematima_inversa(pose_destino: Pose, epsilon: np.float64 = 1e-3, max_iteraciones: np.int32 = 1000) -> np.ndarray:
  q = np.array([0, 0, 0, 0, 0, 0])

  for _ in range(max_iteraciones):
    if funcion_costo(irb140.cinematica_directa(q), pose_destino) < epsilon:
      return q

    # Hacer algo

  print("Llegó a max iteraciones")
  return q

pose_destino = irb140.cinematica_directa(np.array([0, -np.pi/2, np.pi, 0, np.pi/2, 0]))
cinematima_inversa(pose_destino)

Llegó a max iteraciones


array([0, 0, 0, 0, 0, 0])